# Radar Financeiro — todos os clientes

Execute em ordem. Cada leitura e cada etapa Spark registra seu próprio tempo. Q3 lê a réplica Hive; Q1, Q2, Q4 e Q5 leem DB2.


In [ ]:
from traceback import format_exc
try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal
    gerenciador_local = GerenciadorLocal(
        nome_sessao='radar-financeiro-todos',
        exibir_configuracao=False,
        ativar_logs=True,
    )
    spark = gerenciador_local.criar_sessao_spark(db2=True)
    print('[RADAR] Sessão Spark inicializada pelo padrão corporativo.')
except Exception:
    print(format_exc())
    raise


In [ ]:
# Utilitários corporativos; o kernel local e o Spark remoto permanecem separados.
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


In [ ]:
%%spark
import calendar
import datetime
import os
import time
from datetime import timedelta
from decimal import Decimal, ROUND_HALF_UP, localcontext
from pyspark.sql import Window, functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, ShortType,
    StringType, DateType, TimestampType, DecimalType, ArrayType,
)
from pyspark.storagelevel import StorageLevel

periodo = 1
FETCHSIZE = 10000
DATA_EXECUCAO = datetime.date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])
assert type(periodo) is int and 1 <= periodo <= 6
mes_anterior = DATA_EXECUCAO.replace(day=1) - timedelta(days=1)
DATA_INICIAL_PUBLICO = mes_anterior.replace(day=min(DATA_EXECUCAO.day, calendar.monthrange(mes_anterior.year, mes_anterior.month)[1]))
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))


## Público e contexto


In [ ]:
%%spark
inicio = time.perf_counter()
# Q1 — Buscar os registros que formam o público.
df_q1_raw = conector_db2.sql(f"""
SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR,
       NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_EXECUCAO} 00:00:00')
""", fetchsize=FETCHSIZE, query_timeout=900)

df_q1_raw = df_q1_raw.persist(StorageLevel.DISK_ONLY)
qt_q1 = df_q1_raw.count()
print(f'Q1 Público: {qt_q1} registros | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
condicao_conta = (
    (F.col('NR_MCA_PCT_OPB') == 999999999)
    & (F.col('CD_PRD') == 6)
    & F.col('NR_AG_TITR').isNotNull()
    & F.col('CD_CT_TITR').isNotNull()
    & (F.trim(F.col('CD_CT_TITR').cast('string')) != '')
)
q1 = df_q1_raw.groupBy('CD_CLI').agg(
    F.max('TS_INCL_TRAN').alias('TS_INCL_TRAN_REF'),
    F.countDistinct('NR_CPF_CNPJ_TITR').alias('QT_CPFS'),
    F.max('NR_CPF_CNPJ_TITR').cast(DecimalType(14, 0)).alias('CD_CPF_MAX'),
    F.countDistinct(F.when(condicao_conta, F.struct('NR_AG_TITR', 'CD_CT_TITR'))).alias('QT_CONTAS'),
    F.max(F.when(condicao_conta, F.col('NR_AG_TITR'))).alias('NR_AG_TITR_MAX'),
    F.max(F.when(condicao_conta, F.col('CD_CT_TITR'))).alias('CD_CT_TITR_MAX'),
)
df_publico = (q1
    .withColumn('FL_CPF_UNICO', F.when(F.col('QT_CPFS') == 1, 'S').otherwise('N'))
    .withColumn('CD_CPF', F.when(F.col('QT_CPFS') == 1, F.col('CD_CPF_MAX')))
    .withColumn('FL_CONTA_ELEGIVEL_UNICA', F.when(F.col('QT_CONTAS') == 1, 'S').otherwise('N'))
    .withColumn('NR_AG_TITR', F.when(F.col('QT_CONTAS') == 1, F.col('NR_AG_TITR_MAX')))
    .withColumn('CD_CT_TITR', F.when(F.col('QT_CONTAS') == 1, F.col('CD_CT_TITR_MAX')))
)
agencia_txt = F.trim(F.col('NR_AG_TITR').cast('string'))
conta_txt = F.trim(F.col('CD_CT_TITR').cast('string'))
conta_significativa = F.regexp_replace(conta_txt, '^0+', '')
conta_significativa = F.when(conta_significativa == '', '0').otherwise(conta_significativa)
conta_valida = (
    F.col('FL_CONTA_ELEGIVEL_UNICA').eqNullSafe(F.lit('S'))
    & agencia_txt.rlike(r'^[0-9]+$')
    & conta_txt.rlike(r'^[0-9]+$')
    & (agencia_txt.cast('long').between(-2147483648, 2147483647))
    & (F.length(conta_significativa) <= 11)
)
df_publico = (df_publico
    .withColumn('CD_UOR_CC_NORM', F.when(conta_valida, agencia_txt.cast('int')))
    .withColumn('NR_CC_NORM', F.when(conta_valida, conta_significativa.cast(DecimalType(11, 0))))
    .select('CD_CLI', 'TS_INCL_TRAN_REF', 'FL_CPF_UNICO', 'CD_CPF',
            'FL_CONTA_ELEGIVEL_UNICA', 'NR_AG_TITR', 'CD_CT_TITR',
            'CD_UOR_CC_NORM', 'NR_CC_NORM'))
df_publico = df_publico.persist(StorageLevel.MEMORY_AND_DISK)
qt_publico = df_publico.count()
assert qt_publico > 0, 'Público vazio.'
df_q1_raw.unpersist()
print(f'Público Spark: {qt_publico} clientes | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
# Q2 — Buscar os registros de ciclo financeiro.
df_ciclo_raw = conector_db2.sql("""
SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
FROM DB2GFP.CT_GRDR_FNCO
""", fetchsize=FETCHSIZE, query_timeout=900).persist(StorageLevel.DISK_ONLY)
qt_q2 = df_ciclo_raw.count()
print(f'Q2 Ciclo: {qt_q2} registros | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
df_contas = (df_publico.filter(F.col('CD_UOR_CC_NORM').isNotNull())
    .select(F.col('CD_UOR_CC_NORM').alias('CD_UOR_CC'), F.col('NR_CC_NORM').alias('NR_CC')).distinct())
w_ciclo = Window.partitionBy('CD_UOR_CC', 'NR_CC').orderBy(F.col('TS_ULT_EXEA_PSQ').desc_nulls_first())
df_ciclo = (df_ciclo_raw.join(df_contas, ['CD_UOR_CC', 'NR_CC'], 'inner')
    .withColumn('_rn', F.row_number().over(w_ciclo))
    .filter(F.col('_rn') == 1)
    .select(F.col('CD_UOR_CC').alias('CD_UOR_CC_NORM'), F.col('NR_CC').alias('NR_CC_NORM'),
            F.col('DD_INC_MM_CLC_BLC'), F.col('TS_ULT_EXEA_PSQ').alias('TS_DD_INC_MM_CLC_BLC_REF')))
df_contexto = (df_publico
    .join(df_ciclo, ['CD_UOR_CC_NORM', 'NR_CC_NORM'], 'left')
    .withColumn('DD_INC_MM_CLC_BLC_FALLBACK',
                F.when(F.col('CD_UOR_CC_NORM').isNull(), F.lit(None).cast('smallint'))
                 .when(F.col('DD_INC_MM_CLC_BLC').isNull(), F.lit(1).cast('smallint'))
                 .otherwise(F.col('DD_INC_MM_CLC_BLC').cast('smallint'))))
df_contexto = df_contexto.persist(StorageLevel.DISK_ONLY)
qt_contexto = df_contexto.count()
assert qt_contexto == qt_publico
assert not df_contexto.filter(F.col('DD_INC_MM_CLC_BLC_FALLBACK').isNotNull() & ~F.col('DD_INC_MM_CLC_BLC_FALLBACK').between(1, 31)).limit(1).count(), 'Dia de ciclo inválido.'
df_ciclo_raw.unpersist()
print(f'Ciclo Spark: {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
# Q3 — Buscar os registros de renda na réplica Hive.
df_renda_raw = spark.sql("""
SELECT NR_CPF_BASE_SRF, DT_INCL_REN_AVLD, VL_REN
FROM DB2DFE.REN_AVLD_PF
""").persist(StorageLevel.DISK_ONLY)
qt_q3 = df_renda_raw.count()
print(f'Q3 Renda (Hive): {qt_q3} registros | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
df_contexto_anterior = df_contexto
w_renda = Window.partitionBy('NR_CPF_BASE_SRF').orderBy(F.col('DT_INCL_REN_AVLD').desc_nulls_last())
df_renda = (df_renda_raw.join(df_publico.select(F.col('CD_CPF').alias('NR_CPF_BASE_SRF')).distinct(), 'NR_CPF_BASE_SRF', 'inner')
    .withColumn('_rn', F.row_number().over(w_renda))
    .filter(F.col('_rn') == 1)
    .select(F.col('NR_CPF_BASE_SRF').cast(DecimalType(14, 0)).alias('CD_CPF'),
            F.col('DT_INCL_REN_AVLD').cast('date').alias('DT_REN_PRES_REF'),
            (F.col('VL_REN') * F.lit(periodo)).cast(DecimalType(17, 2)).alias('VL_REN_PRES')))
df_contexto = df_contexto.join(df_renda, 'CD_CPF', 'left')
df_contexto = df_contexto.persist(StorageLevel.DISK_ONLY)
qt_contexto = df_contexto.count()
assert qt_contexto == qt_publico
df_contexto_anterior.unpersist()
df_renda_raw.unpersist()
print(f'Renda Spark: {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
# Q4 — Buscar os perfis válidos até a data de execução.
df_perfil_raw = conector_db2.sql(f"""
SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI,
       CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
FROM DB2D1D.DVS_GRDR_FNCO_PF
WHERE DT_REF <= DATE('{DATA_EXECUCAO}')
""", fetchsize=FETCHSIZE, query_timeout=900)
df_perfil_raw = df_perfil_raw.persist(StorageLevel.DISK_ONLY)
qt_q4 = df_perfil_raw.count()
print(f'Q4 Perfil: {qt_q4} registros | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
df_contexto_anterior = df_contexto
w_perfil = Window.partitionBy('CD_CLI').orderBy(F.col('DT_REF').desc_nulls_first())
df_perfil_filtrado = df_perfil_raw.join(df_publico.select('CD_CLI').distinct(), 'CD_CLI', 'inner').persist(StorageLevel.DISK_ONLY)
df_perfil_ordenado = df_perfil_filtrado.withColumn('_rn', F.row_number().over(w_perfil))
empate_perfil = (df_perfil_filtrado.groupBy('CD_CLI', 'DT_REF').count()
                 .withColumn('_max_dt', F.max('DT_REF').over(Window.partitionBy('CD_CLI')))
                 .filter((F.col('DT_REF') == F.col('_max_dt')) & (F.col('count') > 1))
                 .limit(1).count())
assert not empate_perfil, 'Perfil empatado na maior DT_REF.'
df_perfil = (df_perfil_ordenado.filter(F.col('_rn') == 1)
    .select('CD_CLI', F.col('DT_REF').alias('DT_REF_PRFL'),
            F.col('CD_MAC_PRFL_CLI').cast('int'), 'NM_MAC_PRFL_CLI',
            F.col('CD_MIC_PRFL_CLI').cast('int'), 'NM_MIC_PRFL_CLI'))
df_contexto = df_contexto.join(df_perfil, 'CD_CLI', 'left')

df_contexto = df_contexto.persist(StorageLevel.DISK_ONLY)
qt_contexto = df_contexto.count()
assert qt_contexto == qt_publico
df_contexto_anterior.unpersist()
df_perfil_raw.unpersist()
df_perfil_filtrado.unpersist()
print(f'Perfil Spark: {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
# Janela financeira: cálculo nativo com clipping do dia em cada mês.
inicio = time.perf_counter()
df_contexto_anterior = df_contexto
ref_month = F.trunc(F.to_date('TS_INCL_TRAN_REF'), 'MM')
dia = F.col('DD_INC_MM_CLC_BLC_FALLBACK').cast('int')
dia_ref = F.least(dia, F.dayofmonth(F.last_day(ref_month)))
candidato = F.date_add(ref_month, dia_ref - 1)
ref_prev_month = F.add_months(ref_month, -1)
inicio_aberto = F.when(F.to_timestamp('TS_INCL_TRAN_REF') >= F.to_timestamp(candidato), candidato).otherwise(
    F.date_add(ref_prev_month, F.least(dia, F.dayofmonth(F.last_day(ref_prev_month))) - 1)
)
dt_fim = F.date_sub(inicio_aberto, 1)
alvo_inicio = F.add_months(F.trunc(inicio_aberto, 'MM'), -periodo)
dt_ini = F.date_add(alvo_inicio, F.least(dia, F.dayofmonth(F.last_day(alvo_inicio))) - 1)
df_contexto = (df_contexto
    .withColumn('DT_REF_INI', F.when(dia.isNull(), F.lit(None).cast('date')).otherwise(dt_ini))
    .withColumn('DT_REF_FIM', F.when(dia.isNull(), F.lit(None).cast('date')).otherwise(dt_fim))
    .withColumn('DT_EXEA', F.lit(DATA_EXECUCAO).cast('date'))
    .withColumn('DT_MES_EXEA', F.lit(DATA_EXECUCAO.replace(day=1)).cast('date'))
    .select('CD_CLI', 'DT_EXEA', 'DT_MES_EXEA', 'TS_INCL_TRAN_REF',
            'FL_CPF_UNICO', 'CD_CPF', 'FL_CONTA_ELEGIVEL_UNICA',
            'TS_DD_INC_MM_CLC_BLC_REF', 'DD_INC_MM_CLC_BLC',
            'DD_INC_MM_CLC_BLC_FALLBACK', 'DT_REN_PRES_REF', 'VL_REN_PRES',
            'DT_REF_PRFL', 'CD_MAC_PRFL_CLI', 'NM_MAC_PRFL_CLI',
            'CD_MIC_PRFL_CLI', 'NM_MIC_PRFL_CLI', 'DT_REF_INI', 'DT_REF_FIM'))
df_contexto = df_contexto.persist(StorageLevel.MEMORY_AND_DISK)
qt_contexto = df_contexto.count()
df_contexto_anterior.unpersist()
if qt_contexto != qt_publico:
    raise RuntimeError(f'Contexto alterou cardinalidade: público={qt_publico}, contexto={qt_contexto}.')
print(f'[CONTEXTO] {qt_contexto} clientes | {time.perf_counter() - inicio:.3f}s')


## Movimentos


In [ ]:
%%spark
limites = df_contexto.agg(F.min('DT_REF_INI').alias('min_ini'), F.max('DT_REF_FIM').alias('max_fim')).first()
dt_min = (limites['min_ini'] - timedelta(days=5)) if limites['min_ini'] else DATA_EXECUCAO
dt_max = (limites['max_fim'] + timedelta(days=5)) if limites['max_fim'] else DATA_EXECUCAO - timedelta(days=1)


In [ ]:
%%spark
inicio = time.perf_counter()
# Q5 — Buscar movimentos para a janela e a reconciliação.
df_mov_raw = conector_db2.sql(f"""
SELECT NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN,
       CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN, NR_MCA_PCT_OPB
FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND DT_TRAN >= DATE('{dt_min}')
  AND DT_TRAN <= DATE('{dt_max}')
  AND (CD_NTZ_CTB_TRAN = 'C' OR (CD_NTZ_CTB_TRAN = 'D' AND IN_VSLO_CSM = 'S'))
""", fetchsize=FETCHSIZE, query_timeout=900)
df_mov_raw = df_mov_raw.persist(StorageLevel.DISK_ONLY)
qt_q5 = df_mov_raw.count()
print(f'Q5 Movimentos: {qt_q5} registros | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
df_mov = (df_mov_raw
    .join(df_contexto.select('CD_CLI', 'DT_REF_INI', 'DT_REF_FIM'), 'CD_CLI', 'inner')
    .filter(F.col('DT_REF_INI').isNotNull() & F.col('DT_TRAN').between(
        F.date_sub(F.col('DT_REF_INI'), 5),
        F.date_add(F.col('DT_REF_FIM'), 5)))
    .withColumn('IN_JANELA', F.when(
        F.col('DT_TRAN').between(F.col('DT_REF_INI'), F.col('DT_REF_FIM')), 'S'
    ).otherwise('N'))
    .select('NR_TRAN_INST_PCT', 'CD_CLI', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
            'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN', 'NR_MCA_PCT_OPB', 'IN_JANELA'))
df_mov = df_mov.persist(StorageLevel.DISK_ONLY)
qt_mov = df_mov.count()
if df_mov.filter(F.col('NR_TRAN_INST_PCT').isNull()).limit(1).count() or df_mov.groupBy('CD_CLI', 'NR_TRAN_INST_PCT').count().filter('count <> 1').limit(1).count():
    raise RuntimeError('NR_TRAN_INST_PCT não é identidade única por cliente na leitura de movimentos.')
print(f'[MOVIMENTOS] {qt_mov} linhas | {time.perf_counter() - inicio:.3f}s')

df_mov_raw.unpersist()


## Reconciliação


In [ ]:
%%spark
# Pareamento máximo entre bancos distintos; a função executa por grupo no Spark.
inicio = time.perf_counter()
SCHEMA_PAR_EXATO = ArrayType(StructType([
    StructField('ID_CREDITO', LongType(), False), StructField('ID_DEBITO', LongType(), False),
]))
def parear_listas_exatas(lista_creditos, lista_debitos):
    creditos = sorted([(int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB']) for r in (lista_creditos or [])], key=lambda x: x[0])
    debitos = sorted([(int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB']) for r in (lista_debitos or [])], key=lambda x: x[0])
    adj = [[j for j, (_, bd) in enumerate(debitos) if bc is not None and bd is not None and bc != bd] for _, bc in creditos]
    por_d = {}; por_c = {}
    for ci0 in range(len(creditos)):
        fila = [ci0]; vistos_c = {ci0}; vistos_d = set(); anterior = {}; livre = None; pos = 0
        while pos < len(fila) and livre is None:
            ci = fila[pos]; pos += 1
            for di in adj[ci]:
                if di in vistos_d: continue
                vistos_d.add(di); anterior[di] = ci
                if di not in por_d: livre = di; break
                proximo = por_d[di]
                if proximo not in vistos_c: vistos_c.add(proximo); fila.append(proximo)
        if livre is None: continue
        di = livre
        while True:
            ci = anterior[di]; anterior_di = por_c.get(ci); por_d[di] = ci; por_c[ci] = di
            if anterior_di is None: break
            di = anterior_di
    return sorted([(creditos[ci][0], debitos[di][0]) for ci, di in por_c.items()])
spark.udf.register('parear_exato_massivo', parear_listas_exatas, SCHEMA_PAR_EXATO)
c = (df_mov.filter((F.col('NR_TRAN_INST_PCT').isNotNull()) & (F.col('CD_NTZ_CTB_TRAN') == 'C') & F.col('DT_TRAN').isNotNull() & F.col('VL_TRAN').isNotNull() & F.col('CD_TIP_MOE_CRR').isNotNull())
     .groupBy('CD_CLI', 'DT_TRAN', 'VL_TRAN', 'CD_TIP_MOE_CRR', 'IN_JANELA')
     .agg(F.collect_list(F.struct('NR_TRAN_INST_PCT', 'NR_MCA_PCT_OPB')).alias('C')))
d = (df_mov.filter((F.col('NR_TRAN_INST_PCT').isNotNull()) & (F.col('CD_NTZ_CTB_TRAN') == 'D') & F.col('DT_TRAN').isNotNull() & F.col('VL_TRAN').isNotNull() & F.col('CD_TIP_MOE_CRR').isNotNull())
     .groupBy('CD_CLI', 'DT_TRAN', 'VL_TRAN', 'CD_TIP_MOE_CRR', 'IN_JANELA')
     .agg(F.collect_list(F.struct('NR_TRAN_INST_PCT', 'NR_MCA_PCT_OPB')).alias('D')))
df_pares_exatos = (c.join(d, ['CD_CLI', 'DT_TRAN', 'VL_TRAN', 'CD_TIP_MOE_CRR', 'IN_JANELA'])
    .select('CD_CLI', 'IN_JANELA', F.explode(F.expr('parear_exato_massivo(C, D)')).alias('p'))
    .select('CD_CLI', 'IN_JANELA', F.col('p.ID_CREDITO').alias('ID_CREDITO'), F.col('p.ID_DEBITO').alias('ID_DEBITO')))
df_pares_exatos = df_pares_exatos.persist(StorageLevel.DISK_ONLY)
qt_pares_exatos = df_pares_exatos.count()
print(f'Reconciliação exata: {qt_pares_exatos} pares | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
df_ids_exatos = (df_pares_exatos.select('CD_CLI', F.col('ID_CREDITO').alias('NR_TRAN_INST_PCT'))
    .unionByName(df_pares_exatos.select('CD_CLI', F.col('ID_DEBITO').alias('NR_TRAN_INST_PCT'))))
df_residual = df_mov.join(df_ids_exatos, ['CD_CLI', 'NR_TRAN_INST_PCT'], 'leftanti').persist(StorageLevel.DISK_ONLY)
qt_residual = df_residual.count()
print(f'Residual: {qt_residual} movimentos | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
# Pareamento de borda preserva quantidade, distância e desempate da regra atual.
inicio = time.perf_counter()
SCHEMA_PAR_BORDA = ArrayType(StructType([
    StructField('NR_TRAN_DENTRO', LongType(), False), StructField('NR_TRAN_FORA', LongType(), False),
    StructField('DT_TRAN_DENTRO', DateType(), False), StructField('DT_TRAN_FORA', DateType(), False),
    StructField('DIF_DIAS', IntegerType(), False),
]))
def parear_listas_borda(lista_dentro, lista_fora):
    dentro = sorted([(r['DT_TRAN'], int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB']) for r in (lista_dentro or [])], key=lambda x: (x[0], x[1]))
    fora = sorted([(r['DT_TRAN'], int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB']) for r in (lista_fora or [])], key=lambda x: (x[0], x[1]))
    vazio = (0, 0, ()); dp = [[vazio for _ in range(len(fora)+1)] for _ in range(len(dentro)+1)]
    for i in range(len(dentro)-1, -1, -1):
        for j in range(len(fora)-1, -1, -1):
            candidatos = [dp[i+1][j], dp[i][j+1]]
            dt_i, id_i, b_i = dentro[i]; dt_j, id_j, b_j = fora[j]
            dif = abs((dt_i - dt_j).days)
            if 1 <= dif <= 5 and b_i is not None and b_j is not None and b_i != b_j:
                n, custo, pares = dp[i+1][j+1]
                candidatos.append((n+1, custo+dif, ((id_i, id_j, dt_i, dt_j, dif),) + pares))
            dp[i][j] = min(candidatos, key=lambda sol: (-sol[0], sol[1], tuple((p[0], p[1]) for p in sol[2])))
    return list(dp[0][0][2])
spark.udf.register('parear_borda_massivo', parear_listas_borda, SCHEMA_PAR_BORDA)
dentro = (df_residual.filter((F.col('IN_JANELA') == 'S') & F.col('DT_TRAN').isNotNull() & F.col('VL_TRAN').isNotNull() & F.col('CD_TIP_MOE_CRR').isNotNull() & F.col('CD_NTZ_CTB_TRAN').isin('C','D'))
    .groupBy('CD_CLI', 'VL_TRAN', 'CD_TIP_MOE_CRR', 'CD_NTZ_CTB_TRAN')
    .agg(F.collect_list(F.struct('DT_TRAN', 'NR_TRAN_INST_PCT', 'NR_MCA_PCT_OPB')).alias('DENTRO')))
fora = (df_residual.filter((F.col('IN_JANELA') == 'N') & F.col('DT_TRAN').isNotNull() & F.col('VL_TRAN').isNotNull() & F.col('CD_TIP_MOE_CRR').isNotNull() & F.col('CD_NTZ_CTB_TRAN').isin('C','D'))
    .withColumn('NTZ_DENTRO', F.when(F.col('CD_NTZ_CTB_TRAN') == 'C', 'D').otherwise('C'))
    .groupBy('CD_CLI', 'VL_TRAN', 'CD_TIP_MOE_CRR', 'NTZ_DENTRO')
    .agg(F.collect_list(F.struct('DT_TRAN', 'NR_TRAN_INST_PCT', 'NR_MCA_PCT_OPB')).alias('FORA')))
df_pares_borda = (dentro.join(fora, ['CD_CLI','VL_TRAN','CD_TIP_MOE_CRR'], 'inner')
    .filter(F.col('CD_NTZ_CTB_TRAN') == F.col('NTZ_DENTRO'))
    .select('CD_CLI', F.explode(F.expr('parear_borda_massivo(DENTRO, FORA)')).alias('p'))
    .select('CD_CLI', 'p.*'))
df_ids_borda = (df_pares_borda.select('CD_CLI', F.col('NR_TRAN_DENTRO').alias('NR_TRAN_INST_PCT'))
    .unionByName(df_pares_borda.select('CD_CLI', F.col('NR_TRAN_FORA').alias('NR_TRAN_INST_PCT'))))
qt_pares_borda = df_pares_borda.persist(StorageLevel.DISK_ONLY).count()
print(f'[RECONCILIAÇÃO BORDA] {qt_pares_borda} pares | {time.perf_counter() - inicio:.3f}s')


In [ ]:
%%spark
inicio = time.perf_counter()
df_ids_consumidos = df_ids_exatos.unionByName(df_ids_borda)
assert not df_ids_consumidos.groupBy('CD_CLI', 'NR_TRAN_INST_PCT').count().filter('count > 1').limit(1).count(), 'Reconciliação reutilizou transação.'
df_efetivas = (df_residual.filter(F.col('IN_JANELA') == 'S')
    .join(df_ids_borda, ['CD_CLI', 'NR_TRAN_INST_PCT'], 'leftanti')
    .persist(StorageLevel.DISK_ONLY))
qt_efetivas = df_efetivas.count()
df_mov.unpersist()
df_residual.unpersist()
print(f'Movimentos efetivos: {qt_efetivas} | {time.perf_counter() - inicio:.2f}s')


## Classificação


In [ ]:
%%spark
inicio = time.perf_counter()
LINHAS_CATEGORIAS = [
    (None, 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    (None, 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'N'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S', 'N'),
]
schema_categorias = StructType([
    StructField('TIPO', StringType(), True),
    StructField('CD_GRUPO', IntegerType(), False),
    StructField('TX_GRUPO', StringType(), False),
    StructField('CD_CATEGORIA', IntegerType(), False),
    StructField('TX_CATEGORIA', StringType(), False),
    StructField('CD_IR', IntegerType(), False),
    StructField('TX_IR', StringType(), False),
    StructField('CD_CLASS_RADAR', IntegerType(), False),
    StructField('TX_CLASS_RADAR', StringType(), False),
    StructField('IN_AGRO', StringType(), False),
    StructField('IN_PARTICIPA_CALCULO', StringType(), False),
    StructField('IN_PARTICIPA_ORCAMENTO', StringType(), False),
])


df_class = spark.createDataFrame(LINHAS_CATEGORIAS, schema_categorias)
df_classificados = (df_efetivas.alias('m')
    .join(df_class.alias('c'), (F.col('m.CD_CTGR_TRAN_OGNL') == F.col('c.CD_CATEGORIA')) & (F.col('m.CD_NTZ_CTB_TRAN') == F.col('c.TIPO')), 'left')
    .select('m.*', F.col('c.CD_CATEGORIA').isNotNull().alias('TEM_CATEGORIA'),
            F.coalesce(F.col('c.TX_CATEGORIA'), F.lit('Sem Categoria')).alias('TX_CATEGORIA'),
            F.coalesce(F.col('c.CD_CLASS_RADAR'), F.lit(0)).cast('int').alias('CD_CLASS_RADAR'),
            F.coalesce(F.col('c.TX_CLASS_RADAR'), F.lit('Outras Entradas')).alias('TX_CLASS_RADAR'),
            F.coalesce(F.col('c.IN_AGRO'), F.lit('N')).alias('IN_AGRO'),
            F.coalesce(F.col('c.IN_PARTICIPA_CALCULO'), F.lit('N')).alias('IN_PARTICIPA_CALCULO'),
            F.coalesce(F.col('c.IN_PARTICIPA_ORCAMENTO'), F.lit('N')).alias('IN_PARTICIPA_ORCAMENTO')))
df_classificados = df_classificados.persist(StorageLevel.DISK_ONLY)
qt_classificados = df_classificados.count()
assert qt_classificados == qt_efetivas
df_efetivas.unpersist()
print(f'Classificação: {time.perf_counter() - inicio:.2f}s')


## Agregação e motor


In [ ]:
%%spark
# Uma agregação por cliente para todos os valores negociais e para a base realizada.
inicio = time.perf_counter()
brl = (F.col('CD_TIP_MOE_CRR') == 'BRL')
cred = brl & (F.col('CD_NTZ_CTB_TRAN') == 'C')
deb = brl & (F.col('CD_NTZ_CTB_TRAN') == 'D')
df_aggs = df_classificados.groupBy('CD_CLI').agg(
    F.when((F.countDistinct('CD_TIP_MOE_CRR') == 1) & (F.max('CD_TIP_MOE_CRR') == 'BRL'), 'S').otherwise('N').alias('FL_SOMENTE_BRL'),
    F.when(F.count(F.when(brl, 1)) == 0, F.lit(None).cast('string')).when(F.sum(F.when(brl & (F.col('IN_AGRO') == 'S'), 1).otherwise(0)) > 0, 'S').otherwise('N').alias('FL_TEM_MOV_AGRO'),
    F.count(F.when(brl, 1)).cast('long').alias('QT_TRANS_TOTAL'),
    F.sum(F.when(cred, 1).when(brl, 0)).cast('long').alias('QT_TRANS_ENT'),
    F.sum(F.when(deb, 1).when(brl, 0)).cast('long').alias('QT_TRANS_SAI'),
    F.coalesce(F.sum(F.when(cred & (F.col('CD_CLASS_RADAR') == 1) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_ENT_REN'),
    F.coalesce(F.sum(F.when(cred & (F.col('CD_CLASS_RADAR') == 2) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_ENT_EST'),
    F.coalesce(F.sum(F.when(cred & (F.col('CD_CLASS_RADAR') == 3) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_ENT_RESG'),
    F.coalesce(F.sum(F.when(cred & (F.col('CD_CLASS_RADAR') == 0) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_ENT_OUT'),
    F.coalesce(F.sum(F.when(cred & (F.col('CD_CLASS_RADAR') == 4) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_ENT_CRED'),
    F.coalesce(F.sum(F.when(deb & (F.col('CD_CLASS_RADAR') == 5) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_SAI_IND'),
    F.coalesce(F.sum(F.when(deb & (F.col('CD_CLASS_RADAR') == 6) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_SAI_ESS'),
    F.coalesce(F.sum(F.when(deb & (F.col('CD_CLASS_RADAR') == 7) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_SAI_NAO_ESS'),
    F.coalesce(F.sum(F.when(deb & (F.col('CD_CLASS_RADAR') == 8) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_SAI_FUT'),
    F.coalesce(F.sum(F.when(deb & (F.col('CD_CLASS_RADAR') == 9) & (F.col('IN_PARTICIPA_CALCULO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_SAI_OBR'),
    F.coalesce(F.sum(F.when(cred & (F.col('IN_PARTICIPA_ORCAMENTO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_ENT_TOTAL'),
    F.coalesce(F.sum(F.when(deb & (F.col('IN_PARTICIPA_ORCAMENTO') == 'S'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('VL_SAI_TOTAL'),
    F.coalesce(F.sum(F.when(cred & F.col('TEM_CATEGORIA'), F.col('VL_TRAN'))), F.lit(0)).cast('decimal(25,2)').alias('ENTRADAS_REALIZADAS'),
)
df_aggs = (df_aggs
    .withColumn('VL_TRANS_ENT', F.col('VL_ENT_TOTAL'))
    .withColumn('VL_TRANS_SAI', F.col('VL_SAI_TOTAL')))
df_base = df_contexto.join(df_aggs, 'CD_CLI', 'left')
df_base = df_base.withColumn('QT_TRANS_TOTAL', F.coalesce(F.col('QT_TRANS_TOTAL'), F.lit(0).cast('long')))
for nome in ['VL_ENT_REN','VL_ENT_EST','VL_ENT_RESG','VL_ENT_OUT','VL_ENT_CRED','VL_ENT_TOTAL','VL_SAI_IND','VL_SAI_ESS','VL_SAI_NAO_ESS','VL_SAI_FUT','VL_SAI_OBR','VL_SAI_TOTAL','VL_TRANS_ENT','VL_TRANS_SAI','ENTRADAS_REALIZADAS']:
    df_base = df_base.withColumn(nome, F.coalesce(F.col(nome), F.lit(0).cast(DecimalType(25,2))))


df_base = df_base.persist(StorageLevel.DISK_ONLY)
qt_base = df_base.count()
assert qt_base == qt_publico
df_classificados.unpersist()
print(f'Agregação: {qt_base} clientes | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
# Motor reutilizado no resultado oficial e nos dois cenários.
def _dividir_6(numerador, denominador):
    if numerador is None or denominador is None or denominador == 0:
        return None
    with localcontext() as contexto:
        contexto.prec = 50
        valor = (numerador / denominador).quantize(Decimal('0.000001'), rounding=ROUND_HALF_UP)
        return None if abs(valor) >= Decimal('1000') else valor

def calcular_motor_v8(agg, base_orcamento, base_percentuais, cd_macro_perfil):
    if not isinstance(agg, dict):
        agg = agg.asDict()
    qt = agg.get('QT_TRANS_TOTAL')
    saida_total = agg.get('VL_SAI_TOTAL')

    vl_res_orc = None
    if base_orcamento is not None and saida_total is not None:
        vl_res_orc = (base_orcamento - saida_total).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

    pc_sai_ent = None
    if qt != 0 and base_orcamento not in (None, Decimal('0')):
        pc_sai_ent = _dividir_6(saida_total, base_orcamento)

    if pc_sai_ent is None:
        faixa = None
    elif Decimal('0.950000') <= pc_sai_ent <= Decimal('1.050000'):
        faixa = 0
    elif Decimal('1.050000') < pc_sai_ent <= Decimal('1.250000'):
        faixa = 1
    elif pc_sai_ent > Decimal('1.250000'):
        faixa = 2
    elif Decimal('0.750000') <= pc_sai_ent < Decimal('0.950000'):
        faixa = 3
    else:
        faixa = 4

    if faixa is None:
        cd_res_orc = tx_res_orc = tx_sts_res = tx_sts_final = None
    elif faixa == 0:
        cd_res_orc, tx_res_orc, tx_sts_res, tx_sts_final = 0, 'Neutro', None, 'Neutro'
    elif faixa in (1, 2):
        cd_res_orc = 2
        tx_res_orc = 'Deficitário'
        tx_sts_res = 'Moderado' if faixa == 1 else 'Acentuado'
        tx_sts_final = 'Deficitário Moderado' if faixa == 1 else 'Deficitário Acentuado'
    else:
        cd_res_orc = 1
        tx_res_orc = 'Superavitário'
        tx_sts_res = 'Moderado' if faixa == 3 else 'Acentuado'
        tx_sts_final = 'Superavitário Moderado' if faixa == 3 else 'Superavitário Acentuado'

    refs = {
        'IND': Decimal('0.750000'),
        'ESS': Decimal('0.500000'),
        'NAO_ESS': Decimal('0.300000'),
        'FUT': Decimal('0.200000'),
        'OBR': Decimal('0.300000'),
    }
    campos_valores = {
        'IND': 'VL_SAI_IND', 'ESS': 'VL_SAI_ESS',
        'NAO_ESS': 'VL_SAI_NAO_ESS', 'FUT': 'VL_SAI_FUT',
        'OBR': 'VL_SAI_OBR',
    }
    percentuais = {
        tema: (
            None if base_percentuais is None or base_percentuais <= 0
            else _dividir_6(agg.get(campo), base_percentuais)
        )
        for tema, campo in campos_valores.items()
    }

    if qt is None or qt == 0 or base_percentuais is None:
        conc = {tema: None for tema in refs}
    elif base_percentuais <= 0:
        conc = {tema: 0 for tema in refs}
    else:
        pc = percentuais
        conc = {
            'IND': 99 if pc['IND'] is not None and pc['IND'] > Decimal('0.750000') else 0,
            'ESS': (
                0 if pc['ESS'] is not None and pc['ESS'] < Decimal('0.500000')
                else 1 if pc['ESS'] is not None and pc['ESS'] < Decimal('0.750000')
                else 2
            ),
            'NAO_ESS': (
                0 if pc['NAO_ESS'] is not None and pc['NAO_ESS'] < Decimal('0.300000')
                else 1 if pc['NAO_ESS'] is not None and pc['NAO_ESS'] < Decimal('0.450000')
                else 2
            ),
            'FUT': (
                0 if pc['FUT'] is not None and pc['FUT'] >= Decimal('0.300000')
                else 1 if pc['FUT'] is not None and pc['FUT'] >= Decimal('0.200000')
                else 2
            ),
            'OBR': (
                0 if pc['OBR'] is not None and pc['OBR'] < Decimal('0.300000')
                else 1 if pc['OBR'] is not None and pc['OBR'] < Decimal('0.450000')
                else 2
            ),
        }

    if qt is None or qt == 0:
        pont_orc = {tema: None for tema in refs}
    else:
        pont_orc = {'IND': 0}
        for tema in ('ESS', 'NAO_ESS', 'OBR'):
            pont_orc[tema] = None if faixa is None else (2 if faixa == 2 else 1 if faixa in (0, 1) else 0)
        pont_orc['FUT'] = None if faixa is None else (2 if faixa == 4 else 1 if faixa in (0, 3) else 0)

    perfil_valido = cd_macro_perfil in (1, 2, 3)
    pont_prfl = {'IND': None if qt is None or qt == 0 else 0}
    if qt is None or qt == 0 or not perfil_valido:
        pont_prfl.update({tema: None for tema in ('ESS', 'NAO_ESS', 'FUT', 'OBR')})
    else:
        pont_prfl.update({
            'ESS': 0 if cd_macro_perfil == 1 else 1,
            'NAO_ESS': 1 if cd_macro_perfil == 1 else 0,
            'FUT': 2 if cd_macro_perfil == 3 else 1 if cd_macro_perfil == 2 else 0,
            'OBR': 2 if cd_macro_perfil == 1 else 0,
        })

    finais = {'IND': conc['IND']}
    for tema in ('ESS', 'NAO_ESS', 'FUT', 'OBR'):
        parcelas = (conc[tema], pont_orc[tema], pont_prfl[tema])
        finais[tema] = None if any(valor is None for valor in parcelas) else sum(parcelas)

    completa = 'S' if all(finais[tema] is not None for tema in ('IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR')) else 'N'
    if completa == 'N':
        pont_max = qt_temas_max = cd_tema = tx_tema = None
    else:
        ordem = ('IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR')
        codigos = {'IND': 1, 'ESS': 2, 'NAO_ESS': 3, 'FUT': 4, 'OBR': 5}
        rotulos = {
            'IND': 'Categorização dos Gastos',
            'ESS': 'Gestão de Orçamento',
            'NAO_ESS': 'Consumo Planejado',
            'FUT': 'Formação de Reserva',
            'OBR': 'Uso Consciente do Crédito',
        }
        pont_max = max(finais.values())
        vencedores = [tema for tema in ordem if finais[tema] == pont_max]
        qt_temas_max = len(vencedores)
        if qt_temas_max > 1:
            cd_tema, tx_tema = 9, 'Empate'
        else:
            tema = vencedores[0]
            cd_tema, tx_tema = codigos[tema], rotulos[tema]

    resultado = {
        'VL_RES_ORC': vl_res_orc,
        'PC_SAI_ENT': pc_sai_ent,
        'CD_RES_ORC': cd_res_orc,
        'TX_RES_ORC': tx_res_orc,
        'CD_FAIXA_ORC': faixa,
        'TX_STS_RES': tx_sts_res,
        'TX_STS_FINAL': tx_sts_final,
    }
    for tema in ('IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR'):
        resultado[f'PC_SAI_{tema}'] = percentuais[tema]
        resultado[f'PC_REF_{tema}'] = refs[tema]
        resultado[f'NR_PONT_CONC_{tema}'] = conc[tema]
        resultado[f'NR_PONT_ORC_{tema}'] = pont_orc[tema]
        resultado[f'NR_PONT_PRFL_{tema}'] = pont_prfl[tema]
        resultado[f'NR_PONT_{tema}_FIM'] = finais[tema]
    resultado.update({
        'FL_PONTUACAO_COMPLETA': completa,
        'NR_PONT_MAX': pont_max,
        'QT_TEMAS_PONT_MAX': qt_temas_max,
        'CD_TEMA_VENCEDOR': cd_tema,
        'TX_TEMA_VENCEDOR': tx_tema,
    })
    return resultado


In [ ]:
%%spark
caso = {'QT_TRANS_TOTAL': 1, 'VL_SAI_TOTAL': Decimal('0.00'),
        'VL_SAI_IND': Decimal('0.00'), 'VL_SAI_ESS': Decimal('0.00'),
        'VL_SAI_NAO_ESS': Decimal('0.00'), 'VL_SAI_FUT': Decimal('0.00'), 'VL_SAI_OBR': Decimal('0.00')}
assert _dividir_6(Decimal('1'), Decimal('128')) == Decimal('0.007813')
for saida, faixa in [('95', 0), ('105', 0), ('105.01', 1), ('125', 1), ('125.01', 2), ('75', 3), ('74.99', 4)]:
    caso['VL_SAI_TOTAL'] = Decimal(saida)
    assert calcular_motor_v8(caso, Decimal('100'), Decimal('100'), 2)['CD_FAIXA_ORC'] == faixa
caso['VL_SAI_TOTAL'] = Decimal('100')
resultado_teste = calcular_motor_v8(caso, Decimal('0'), Decimal('100'), 2)
assert resultado_teste['VL_RES_ORC'] == Decimal('-100') and resultado_teste['PC_SAI_ENT'] is None
caso['QT_TRANS_TOTAL'] = 0
resultado_teste = calcular_motor_v8(caso, None, None, None)
assert resultado_teste['VL_RES_ORC'] is None and resultado_teste['FL_PONTUACAO_COMPLETA'] == 'N'
print('Casos essenciais do motor aprovados.')


In [ ]:
%%spark
schema_motor = StructType([
    StructField('VL_RES_ORC', DecimalType(25, 2), True),
    StructField('PC_SAI_ENT', DecimalType(9, 6), True),
    StructField('CD_RES_ORC', IntegerType(), True),
    StructField('TX_RES_ORC', StringType(), True),
    StructField('CD_FAIXA_ORC', IntegerType(), True),
    StructField('TX_STS_RES', StringType(), True),
    StructField('TX_STS_FINAL', StringType(), True),
    StructField('PC_SAI_IND', DecimalType(9, 6), True),
    StructField('PC_SAI_ESS', DecimalType(9, 6), True),
    StructField('PC_SAI_NAO_ESS', DecimalType(9, 6), True),
    StructField('PC_SAI_FUT', DecimalType(9, 6), True),
    StructField('PC_SAI_OBR', DecimalType(9, 6), True),
    StructField('PC_REF_IND', DecimalType(9, 6), True),
    StructField('PC_REF_ESS', DecimalType(9, 6), True),
    StructField('PC_REF_NAO_ESS', DecimalType(9, 6), True),
    StructField('PC_REF_FUT', DecimalType(9, 6), True),
    StructField('PC_REF_OBR', DecimalType(9, 6), True),
    StructField('NR_PONT_CONC_IND', IntegerType(), True),
    StructField('NR_PONT_CONC_ESS', IntegerType(), True),
    StructField('NR_PONT_CONC_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_CONC_FUT', IntegerType(), True),
    StructField('NR_PONT_CONC_OBR', IntegerType(), True),
    StructField('NR_PONT_ORC_IND', IntegerType(), True),
    StructField('NR_PONT_ORC_ESS', IntegerType(), True),
    StructField('NR_PONT_ORC_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_ORC_FUT', IntegerType(), True),
    StructField('NR_PONT_ORC_OBR', IntegerType(), True),
    StructField('NR_PONT_PRFL_IND', IntegerType(), True),
    StructField('NR_PONT_PRFL_ESS', IntegerType(), True),
    StructField('NR_PONT_PRFL_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_PRFL_FUT', IntegerType(), True),
    StructField('NR_PONT_PRFL_OBR', IntegerType(), True),
    StructField('NR_PONT_IND_FIM', IntegerType(), True),
    StructField('NR_PONT_ESS_FIM', IntegerType(), True),
    StructField('NR_PONT_NAO_ESS_FIM', IntegerType(), True),
    StructField('NR_PONT_FUT_FIM', IntegerType(), True),
    StructField('NR_PONT_OBR_FIM', IntegerType(), True),
    StructField('FL_PONTUACAO_COMPLETA', StringType(), True),
    StructField('NR_PONT_MAX', IntegerType(), True),
    StructField('QT_TEMAS_PONT_MAX', IntegerType(), True),
    StructField('CD_TEMA_VENCEDOR', IntegerType(), True),
    StructField('TX_TEMA_VENCEDOR', StringType(), True),
])
motor = F.udf(calcular_motor_v8, schema_motor)


In [ ]:
%%spark
inicio = time.perf_counter()
valores_motor = F.struct('QT_TRANS_TOTAL', 'VL_SAI_TOTAL', 'VL_SAI_IND', 'VL_SAI_ESS', 'VL_SAI_NAO_ESS', 'VL_SAI_FUT', 'VL_SAI_OBR')
tem_janela = F.col('DT_REF_INI').isNotNull() & F.col('DT_REF_FIM').isNotNull()
base_renda = F.when(tem_janela, F.col('VL_REN_PRES'))
base_realizada = F.when(tem_janela, F.col('ENTRADAS_REALIZADAS'))
df_motor = (df_base
    .withColumn('oficial', motor(valores_motor, F.col('VL_ENT_TOTAL'), F.col('VL_REN_PRES'), F.col('CD_MAC_PRFL_CLI')))
    .withColumn('renda', motor(valores_motor, base_renda, base_renda, F.col('CD_MAC_PRFL_CLI')))
    .withColumn('realizada', motor(valores_motor, base_realizada, base_realizada, F.col('CD_MAC_PRFL_CLI'))))
df_final = df_motor.select(
    F.col('CD_CLI').cast('int').alias('CD_CLI'),
    F.col('DT_EXEA').cast('date').alias('DT_EXEA'),
    F.col('DT_MES_EXEA').cast('date').alias('DT_MES_EXEA'),
    F.col('TS_INCL_TRAN_REF').cast('timestamp').alias('TS_INCL_TRAN_REF'),
    F.col('FL_CPF_UNICO').cast('string').alias('FL_CPF_UNICO'),
    F.col('CD_CPF').cast('decimal(14,0)').alias('CD_CPF'),
    F.col('FL_CONTA_ELEGIVEL_UNICA').cast('string').alias('FL_CONTA_ELEGIVEL_UNICA'),
    F.col('TS_DD_INC_MM_CLC_BLC_REF').cast('timestamp').alias('TS_DD_INC_MM_CLC_BLC_REF'),
    F.col('DD_INC_MM_CLC_BLC').cast('smallint').alias('DD_INC_MM_CLC_BLC'),
    F.col('DD_INC_MM_CLC_BLC_FALLBACK').cast('smallint').alias('DD_INC_MM_CLC_BLC_FALLBACK'),
    F.col('DT_REN_PRES_REF').cast('date').alias('DT_REN_PRES_REF'),
    F.col('VL_REN_PRES').cast('decimal(17,2)').alias('VL_REN_PRES'),
    F.col('DT_REF_PRFL').cast('date').alias('DT_REF_PRFL'),
    F.col('CD_MAC_PRFL_CLI').cast('int').alias('CD_MAC_PRFL_CLI'),
    F.col('NM_MAC_PRFL_CLI').cast('string').alias('NM_MAC_PRFL_CLI'),
    F.col('CD_MIC_PRFL_CLI').cast('int').alias('CD_MIC_PRFL_CLI'),
    F.col('NM_MIC_PRFL_CLI').cast('string').alias('NM_MIC_PRFL_CLI'),
    F.col('DT_REF_INI').cast('date').alias('DT_REF_INI'),
    F.col('DT_REF_FIM').cast('date').alias('DT_REF_FIM'),
    F.col('FL_SOMENTE_BRL').cast('string').alias('FL_SOMENTE_BRL'),
    F.col('FL_TEM_MOV_AGRO').cast('string').alias('FL_TEM_MOV_AGRO'),
    F.col('QT_TRANS_TOTAL').cast('bigint').alias('QT_TRANS_TOTAL'),
    F.col('QT_TRANS_ENT').cast('bigint').alias('QT_TRANS_ENT'),
    F.col('QT_TRANS_SAI').cast('bigint').alias('QT_TRANS_SAI'),
    F.col('VL_TRANS_ENT').cast('decimal(25,2)').alias('VL_TRANS_ENT'),
    F.col('VL_TRANS_SAI').cast('decimal(25,2)').alias('VL_TRANS_SAI'),
    F.col('VL_ENT_REN').cast('decimal(25,2)').alias('VL_ENT_REN'),
    F.col('VL_ENT_EST').cast('decimal(25,2)').alias('VL_ENT_EST'),
    F.col('VL_ENT_RESG').cast('decimal(25,2)').alias('VL_ENT_RESG'),
    F.col('VL_ENT_OUT').cast('decimal(25,2)').alias('VL_ENT_OUT'),
    F.col('VL_ENT_CRED').cast('decimal(25,2)').alias('VL_ENT_CRED'),
    F.col('VL_ENT_TOTAL').cast('decimal(25,2)').alias('VL_ENT_TOTAL'),
    F.col('VL_SAI_IND').cast('decimal(25,2)').alias('VL_SAI_IND'),
    F.col('VL_SAI_ESS').cast('decimal(25,2)').alias('VL_SAI_ESS'),
    F.col('VL_SAI_NAO_ESS').cast('decimal(25,2)').alias('VL_SAI_NAO_ESS'),
    F.col('VL_SAI_FUT').cast('decimal(25,2)').alias('VL_SAI_FUT'),
    F.col('VL_SAI_OBR').cast('decimal(25,2)').alias('VL_SAI_OBR'),
    F.col('VL_SAI_TOTAL').cast('decimal(25,2)').alias('VL_SAI_TOTAL'),
    F.col('oficial.VL_RES_ORC').cast('decimal(25,2)').alias('VL_RES_ORC'),
    F.col('oficial.PC_SAI_ENT').cast('decimal(9,6)').alias('PC_SAI_ENT'),
    F.col('oficial.CD_RES_ORC').cast('int').alias('CD_RES_ORC'),
    F.col('oficial.TX_RES_ORC').cast('string').alias('TX_RES_ORC'),
    F.col('oficial.CD_FAIXA_ORC').cast('int').alias('CD_FAIXA_ORC'),
    F.col('oficial.TX_STS_RES').cast('string').alias('TX_STS_RES'),
    F.col('oficial.TX_STS_FINAL').cast('string').alias('TX_STS_FINAL'),
    F.col('oficial.PC_SAI_IND').cast('decimal(9,6)').alias('PC_SAI_IND'),
    F.col('oficial.PC_SAI_ESS').cast('decimal(9,6)').alias('PC_SAI_ESS'),
    F.col('oficial.PC_SAI_NAO_ESS').cast('decimal(9,6)').alias('PC_SAI_NAO_ESS'),
    F.col('oficial.PC_SAI_FUT').cast('decimal(9,6)').alias('PC_SAI_FUT'),
    F.col('oficial.PC_SAI_OBR').cast('decimal(9,6)').alias('PC_SAI_OBR'),
    F.col('oficial.PC_REF_IND').cast('decimal(9,6)').alias('PC_REF_IND'),
    F.col('oficial.PC_REF_ESS').cast('decimal(9,6)').alias('PC_REF_ESS'),
    F.col('oficial.PC_REF_NAO_ESS').cast('decimal(9,6)').alias('PC_REF_NAO_ESS'),
    F.col('oficial.PC_REF_FUT').cast('decimal(9,6)').alias('PC_REF_FUT'),
    F.col('oficial.PC_REF_OBR').cast('decimal(9,6)').alias('PC_REF_OBR'),
    F.col('oficial.NR_PONT_CONC_IND').cast('int').alias('NR_PONT_CONC_IND'),
    F.col('oficial.NR_PONT_CONC_ESS').cast('int').alias('NR_PONT_CONC_ESS'),
    F.col('oficial.NR_PONT_CONC_NAO_ESS').cast('int').alias('NR_PONT_CONC_NAO_ESS'),
    F.col('oficial.NR_PONT_CONC_FUT').cast('int').alias('NR_PONT_CONC_FUT'),
    F.col('oficial.NR_PONT_CONC_OBR').cast('int').alias('NR_PONT_CONC_OBR'),
    F.col('oficial.NR_PONT_ORC_IND').cast('int').alias('NR_PONT_ORC_IND'),
    F.col('oficial.NR_PONT_ORC_ESS').cast('int').alias('NR_PONT_ORC_ESS'),
    F.col('oficial.NR_PONT_ORC_NAO_ESS').cast('int').alias('NR_PONT_ORC_NAO_ESS'),
    F.col('oficial.NR_PONT_ORC_FUT').cast('int').alias('NR_PONT_ORC_FUT'),
    F.col('oficial.NR_PONT_ORC_OBR').cast('int').alias('NR_PONT_ORC_OBR'),
    F.col('oficial.NR_PONT_PRFL_IND').cast('int').alias('NR_PONT_PRFL_IND'),
    F.col('oficial.NR_PONT_PRFL_ESS').cast('int').alias('NR_PONT_PRFL_ESS'),
    F.col('oficial.NR_PONT_PRFL_NAO_ESS').cast('int').alias('NR_PONT_PRFL_NAO_ESS'),
    F.col('oficial.NR_PONT_PRFL_FUT').cast('int').alias('NR_PONT_PRFL_FUT'),
    F.col('oficial.NR_PONT_PRFL_OBR').cast('int').alias('NR_PONT_PRFL_OBR'),
    F.col('oficial.NR_PONT_IND_FIM').cast('int').alias('NR_PONT_IND_FIM'),
    F.col('oficial.NR_PONT_ESS_FIM').cast('int').alias('NR_PONT_ESS_FIM'),
    F.col('oficial.NR_PONT_NAO_ESS_FIM').cast('int').alias('NR_PONT_NAO_ESS_FIM'),
    F.col('oficial.NR_PONT_FUT_FIM').cast('int').alias('NR_PONT_FUT_FIM'),
    F.col('oficial.NR_PONT_OBR_FIM').cast('int').alias('NR_PONT_OBR_FIM'),
    F.col('oficial.FL_PONTUACAO_COMPLETA').cast('string').alias('FL_PONTUACAO_COMPLETA'),
    F.col('oficial.NR_PONT_MAX').cast('int').alias('NR_PONT_MAX'),
    F.col('oficial.QT_TEMAS_PONT_MAX').cast('int').alias('QT_TEMAS_PONT_MAX'),
    F.col('oficial.CD_TEMA_VENCEDOR').cast('int').alias('CD_TEMA_VENCEDOR'),
    F.col('oficial.TX_TEMA_VENCEDOR').cast('string').alias('TX_TEMA_VENCEDOR'),
    F.col('renda.VL_RES_ORC').alias('VL_RES_ORC_RENDA_PRESUMIDA'),
    F.col('renda.PC_SAI_ENT').alias('PC_SAI_ENT_RENDA_PRESUMIDA'),
    F.col('renda.CD_RES_ORC').alias('CD_RES_ORC_RENDA_PRESUMIDA'),
    F.col('renda.TX_RES_ORC').alias('TX_RES_ORC_RENDA_PRESUMIDA'),
    F.col('renda.CD_FAIXA_ORC').alias('CD_FAIXA_ORC_RENDA_PRESUMIDA'),
    F.col('renda.TX_STS_RES').alias('TX_STS_RES_RENDA_PRESUMIDA'),
    F.col('renda.TX_STS_FINAL').alias('TX_STS_FINAL_RENDA_PRESUMIDA'),
    F.col('renda.PC_SAI_IND').alias('PC_SAI_IND_RENDA_PRESUMIDA'),
    F.col('renda.PC_SAI_ESS').alias('PC_SAI_ESS_RENDA_PRESUMIDA'),
    F.col('renda.PC_SAI_NAO_ESS').alias('PC_SAI_NAO_ESS_RENDA_PRESUMIDA'),
    F.col('renda.PC_SAI_FUT').alias('PC_SAI_FUT_RENDA_PRESUMIDA'),
    F.col('renda.PC_SAI_OBR').alias('PC_SAI_OBR_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_CONC_IND').alias('NR_PONT_CONC_IND_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_CONC_ESS').alias('NR_PONT_CONC_ESS_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_CONC_NAO_ESS').alias('NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_CONC_FUT').alias('NR_PONT_CONC_FUT_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_CONC_OBR').alias('NR_PONT_CONC_OBR_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_ORC_ESS').alias('NR_PONT_ORC_ESS_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_ORC_NAO_ESS').alias('NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_ORC_FUT').alias('NR_PONT_ORC_FUT_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_ORC_OBR').alias('NR_PONT_ORC_OBR_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_IND_FIM').alias('NR_PONT_IND_FIM_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_ESS_FIM').alias('NR_PONT_ESS_FIM_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_NAO_ESS_FIM').alias('NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_FUT_FIM').alias('NR_PONT_FUT_FIM_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_OBR_FIM').alias('NR_PONT_OBR_FIM_RENDA_PRESUMIDA'),
    F.col('renda.FL_PONTUACAO_COMPLETA').alias('FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA'),
    F.col('renda.NR_PONT_MAX').alias('NR_PONT_MAX_RENDA_PRESUMIDA'),
    F.col('renda.QT_TEMAS_PONT_MAX').alias('QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA'),
    F.col('renda.CD_TEMA_VENCEDOR').alias('CD_TEMA_VENCEDOR_RENDA_PRESUMIDA'),
    F.col('renda.TX_TEMA_VENCEDOR').alias('TX_TEMA_VENCEDOR_RENDA_PRESUMIDA'),
    F.col('realizada.VL_RES_ORC').alias('VL_RES_ORC_ENTRADAS_REALIZADAS'),
    F.col('realizada.PC_SAI_ENT').alias('PC_SAI_ENT_ENTRADAS_REALIZADAS'),
    F.col('realizada.CD_RES_ORC').alias('CD_RES_ORC_ENTRADAS_REALIZADAS'),
    F.col('realizada.TX_RES_ORC').alias('TX_RES_ORC_ENTRADAS_REALIZADAS'),
    F.col('realizada.CD_FAIXA_ORC').alias('CD_FAIXA_ORC_ENTRADAS_REALIZADAS'),
    F.col('realizada.TX_STS_RES').alias('TX_STS_RES_ENTRADAS_REALIZADAS'),
    F.col('realizada.TX_STS_FINAL').alias('TX_STS_FINAL_ENTRADAS_REALIZADAS'),
    F.col('realizada.PC_SAI_IND').alias('PC_SAI_IND_ENTRADAS_REALIZADAS'),
    F.col('realizada.PC_SAI_ESS').alias('PC_SAI_ESS_ENTRADAS_REALIZADAS'),
    F.col('realizada.PC_SAI_NAO_ESS').alias('PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS'),
    F.col('realizada.PC_SAI_FUT').alias('PC_SAI_FUT_ENTRADAS_REALIZADAS'),
    F.col('realizada.PC_SAI_OBR').alias('PC_SAI_OBR_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_CONC_IND').alias('NR_PONT_CONC_IND_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_CONC_ESS').alias('NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_CONC_NAO_ESS').alias('NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_CONC_FUT').alias('NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_CONC_OBR').alias('NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_ORC_ESS').alias('NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_ORC_NAO_ESS').alias('NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_ORC_FUT').alias('NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_ORC_OBR').alias('NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_IND_FIM').alias('NR_PONT_IND_FIM_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_ESS_FIM').alias('NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_NAO_ESS_FIM').alias('NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_FUT_FIM').alias('NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_OBR_FIM').alias('NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS'),
    F.col('realizada.FL_PONTUACAO_COMPLETA').alias('FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS'),
    F.col('realizada.NR_PONT_MAX').alias('NR_PONT_MAX_ENTRADAS_REALIZADAS'),
    F.col('realizada.QT_TEMAS_PONT_MAX').alias('QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS'),
    F.col('realizada.CD_TEMA_VENCEDOR').alias('CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS'),
    F.col('realizada.TX_TEMA_VENCEDOR').alias('TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS'),
).persist(StorageLevel.DISK_ONLY)
qt_final = df_final.count()
print(f'Motor: {qt_final} clientes | {time.perf_counter() - inicio:.2f}s')


In [ ]:
%%spark
assert len(df_final.columns) == 142, 'Resultado deve possuir 142 colunas.'
assert qt_final == qt_publico, 'Quantidade de clientes alterada.'
assert df_final.select('CD_CLI').distinct().count() == qt_final, 'CD_CLI duplicado.'
print('Resultado validado.')


## Publicação


In [ ]:
%%spark
DDL_ANA_RADAR_FIN_CLI = """CREATE TABLE ANA_RADAR_FIN_CLI (

    -- ============================================================
    -- IDENTIFICAÇÃO, REFERÊNCIAS E CONTEXTO
    -- ============================================================

    CD_CLI INT
        COMMENT 'Código identificador do cliente processado pelo Radar',

    DT_EXEA DATE
        COMMENT 'Data de execução do cálculo do Radar',

    DT_MES_EXEA DATE
        COMMENT 'Primeiro dia do mês correspondente à data de execução',

    TS_INCL_TRAN_REF TIMESTAMP
        COMMENT 'Maior timestamp de inclusão de transação utilizado como referência para formação da janela financeira',

    FL_CPF_UNICO STRING
        COMMENT 'Indica se existe exatamente um CPF distinto associado ao cliente no público analisado: S=sim; N=não',

    CD_CPF DECIMAL(14,0)
        COMMENT 'Código do CPF único associado ao cliente; nulo quando não existe unicidade',

    FL_CONTA_ELEGIVEL_UNICA STRING
        COMMENT 'Indica se existe exatamente uma conta elegível para determinação do ciclo financeiro: S=sim; N=não',

    TS_DD_INC_MM_CLC_BLC_REF TIMESTAMP
        COMMENT 'Timestamp de referência do registro de ciclo financeiro selecionado',

    DD_INC_MM_CLC_BLC SMALLINT
        COMMENT 'Dia de início do ciclo financeiro obtido da fonte',

    DD_INC_MM_CLC_BLC_FALLBACK SMALLINT
        COMMENT 'Dia de ciclo efetivamente utilizado após aplicação da regra de fallback',

    DT_REN_PRES_REF DATE
        COMMENT 'Data de referência da renda presumida selecionada',

    VL_REN_PRES DECIMAL(17,2)
        COMMENT 'Valor da renda presumida utilizada pelo Radar',

    DT_REF_PRFL DATE
        COMMENT 'Data de referência do perfil financeiro selecionado',

    CD_MAC_PRFL_CLI INT
        COMMENT 'Código do macroperfil financeiro recebido da fonte; o notebook usa 1, 2 e 3 na pontuação, sem definir rótulos negociais próprios',

    NM_MAC_PRFL_CLI STRING
        COMMENT 'Nome do macroperfil financeiro recebido da fonte',

    CD_MIC_PRFL_CLI INT
        COMMENT 'Código do microperfil financeiro recebido da fonte; o motor atual não cria um de-para negocial próprio para este código',

    NM_MIC_PRFL_CLI STRING
        COMMENT 'Nome do microperfil financeiro recebido da fonte',

    DT_REF_INI DATE
        COMMENT 'Data inicial da janela financeira analisada',

    DT_REF_FIM DATE
        COMMENT 'Data final da janela financeira analisada',

    FL_SOMENTE_BRL STRING
        COMMENT 'Calculado sobre os movimentos efetivos: S quando a única moeda distinta é BRL; N nos demais casos; NULL quando não há movimentos efetivos',

    FL_TEM_MOV_AGRO STRING
        COMMENT 'Indica movimentação agro entre os movimentos efetivos BRL: S=possui movimento agro; N=não possui; NULL=sem movimentos BRL',


    -- ============================================================
    -- MOVIMENTAÇÃO
    -- ============================================================

    QT_TRANS_TOTAL BIGINT
        COMMENT 'Quantidade total de movimentos efetivos BRL considerados na janela',

    QT_TRANS_ENT BIGINT
        COMMENT 'Quantidade de movimentos efetivos de entrada em BRL',

    QT_TRANS_SAI BIGINT
        COMMENT 'Quantidade de movimentos efetivos de saída em BRL',

    VL_TRANS_ENT DECIMAL(25,2)
        COMMENT 'Valor das entradas participantes do cálculo; no contrato atual corresponde ao VL_ENT_TOTAL',

    VL_TRANS_SAI DECIMAL(25,2)
        COMMENT 'Valor das saídas participantes do cálculo; no contrato atual corresponde ao VL_SAI_TOTAL',


    -- ============================================================
    -- ENTRADAS
    -- ============================================================

    VL_ENT_REN DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Renda, classe 1',

    VL_ENT_EST DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Estorno, classe 2',

    VL_ENT_RESG DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Resgate, classe 3',

    VL_ENT_OUT DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Outras Entradas, classe 0',

    VL_ENT_CRED DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Crédito, classe 4',

    VL_ENT_TOTAL DECIMAL(25,2)
        COMMENT 'Valor total das entradas participantes do orçamento',


    -- ============================================================
    -- SAÍDAS
    -- ============================================================

    VL_SAI_IND DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Indeterminadas, classe 5',

    VL_SAI_ESS DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Essenciais, classe 6',

    VL_SAI_NAO_ESS DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Não Essenciais, classe 7',

    VL_SAI_FUT DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Futuro, classe 8',

    VL_SAI_OBR DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Obrigações, classe 9',

    VL_SAI_TOTAL DECIMAL(25,2)
        COMMENT 'Valor total das saídas participantes do orçamento',


    -- ============================================================
    -- RESULTADO ORÇAMENTÁRIO OFICIAL
    -- ============================================================

    VL_RES_ORC DECIMAL(25,2)
        COMMENT 'Resultado orçamentário calculado pela diferença entre VL_ENT_TOTAL e VL_SAI_TOTAL',

    PC_SAI_ENT DECIMAL(9,6)
        COMMENT 'Relação entre o total de saídas e a base de entrada utilizada no orçamento oficial',

    CD_RES_ORC INT
        COMMENT 'Código do resultado orçamentário: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC STRING
        COMMENT 'Descrição do resultado orçamentário correspondente ao CD_RES_ORC',

    CD_FAIXA_ORC INT
        COMMENT 'Código da faixa orçamentária: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES STRING
        COMMENT 'Intensidade do resultado orçamentário: Moderado ou Acentuado; NULL para resultado Neutro ou não calculado',

    TX_STS_FINAL STRING
        COMMENT 'Descrição final do resultado: Neutro, Deficitário Moderado, Deficitário Acentuado, Superavitário Moderado ou Superavitário Acentuado',


    -- ============================================================
    -- PERCENTUAIS OFICIAIS
    -- ============================================================

    PC_SAI_IND DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas em relação à renda presumida no resultado oficial',

    PC_SAI_ESS DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais em relação à renda presumida no resultado oficial',

    PC_SAI_NAO_ESS DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais em relação à renda presumida no resultado oficial',

    PC_SAI_FUT DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro em relação à renda presumida no resultado oficial',

    PC_SAI_OBR DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações em relação à renda presumida no resultado oficial',

    PC_REF_IND DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Indeterminada: 0.750000 corresponde a 75%',

    PC_REF_ESS DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Essencial: 0.500000 corresponde a 50%',

    PC_REF_NAO_ESS DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Não Essencial: 0.300000 corresponde a 30%',

    PC_REF_FUT DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Futuro: 0.200000 corresponde a 20%',

    PC_REF_OBR DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Obrigações: 0.300000 corresponde a 30%',


    -- ============================================================
    -- PONTUAÇÃO DE CONCENTRAÇÃO
    -- ============================================================

    NR_PONT_CONC_IND INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada; valores produzidos pelo motor incluem 0 e 99',

    NR_PONT_CONC_ESS INT
        COMMENT 'Pontuação de concentração da categoria Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_NAO_ESS INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_FUT INT
        COMMENT 'Pontuação de concentração da categoria Futuro; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_OBR INT
        COMMENT 'Pontuação de concentração da categoria Obrigações; valores produzidos pelo motor: 0, 1 ou 2',


    -- ============================================================
    -- PONTUAÇÃO ORÇAMENTÁRIA
    -- ============================================================

    NR_PONT_ORC_IND INT
        COMMENT 'Pontuação orçamentária da categoria Indeterminada',

    NR_PONT_ORC_ESS INT
        COMMENT 'Pontuação orçamentária da categoria Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_NAO_ESS INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_FUT INT
        COMMENT 'Pontuação orçamentária da categoria Futuro; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_OBR INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações; valores produzidos pelo motor: 0, 1 ou 2',


    -- ============================================================
    -- PONTUAÇÃO DE PERFIL
    -- ============================================================

    NR_PONT_PRFL_IND INT
        COMMENT 'Pontuação de perfil da categoria Indeterminada',

    NR_PONT_PRFL_ESS INT
        COMMENT 'Pontuação de perfil da categoria Essencial',

    NR_PONT_PRFL_NAO_ESS INT
        COMMENT 'Pontuação de perfil da categoria Não Essencial',

    NR_PONT_PRFL_FUT INT
        COMMENT 'Pontuação de perfil da categoria Futuro',

    NR_PONT_PRFL_OBR INT
        COMMENT 'Pontuação de perfil da categoria Obrigações',


    -- ============================================================
    -- PONTUAÇÃO FINAL E TEMA VENCEDOR
    -- ============================================================

    NR_PONT_IND_FIM INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos',

    NR_PONT_ESS_FIM INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento',

    NR_PONT_NAO_ESS_FIM INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado',

    NR_PONT_FUT_FIM INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva',

    NR_PONT_OBR_FIM INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito',

    FL_PONTUACAO_COMPLETA STRING
        COMMENT 'Indica se as cinco pontuações finais foram calculadas: S=completa; N=incompleta',

    NR_PONT_MAX INT
        COMMENT 'Maior pontuação final obtida entre os cinco temas',

    QT_TEMAS_PONT_MAX INT
        COMMENT 'Quantidade de temas que possuem a maior pontuação final; exemplo: 1 indica vencedor único e valor maior que 1 indica empate',

    CD_TEMA_VENCEDOR INT
        COMMENT 'Código do tema vencedor: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR STRING
        COMMENT 'Descrição correspondente ao CD_TEMA_VENCEDOR',


    -- ============================================================
    -- CENÁRIO RENDA PRESUMIDA
    -- ============================================================

    VL_RES_ORC_RENDA_PRESUMIDA DECIMAL(25,2)
        COMMENT 'Resultado orçamentário recalculado no cenário RENDA_PRESUMIDA',

    PC_SAI_ENT_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Relação entre saídas e base utilizada no cenário RENDA_PRESUMIDA',

    CD_RES_ORC_RENDA_PRESUMIDA INT
        COMMENT 'Código do resultado orçamentário no cenário RENDA_PRESUMIDA: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição do resultado orçamentário no cenário RENDA_PRESUMIDA',

    CD_FAIXA_ORC_RENDA_PRESUMIDA INT
        COMMENT 'Código da faixa orçamentária no cenário RENDA_PRESUMIDA: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES_RENDA_PRESUMIDA STRING
        COMMENT 'Intensidade do resultado orçamentário no cenário RENDA_PRESUMIDA',

    TX_STS_FINAL_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição final do resultado orçamentário no cenário RENDA_PRESUMIDA',

    PC_SAI_IND_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas no cenário RENDA_PRESUMIDA',

    PC_SAI_ESS_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais no cenário RENDA_PRESUMIDA',

    PC_SAI_NAO_ESS_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais no cenário RENDA_PRESUMIDA',

    PC_SAI_FUT_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro no cenário RENDA_PRESUMIDA',

    PC_SAI_OBR_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_IND_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_FUT_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Futuro no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_OBR_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_FUT_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Futuro no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_OBR_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_IND_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos, no cenário RENDA_PRESUMIDA',

    NR_PONT_ESS_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento, no cenário RENDA_PRESUMIDA',

    NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado, no cenário RENDA_PRESUMIDA',

    NR_PONT_FUT_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva, no cenário RENDA_PRESUMIDA',

    NR_PONT_OBR_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito, no cenário RENDA_PRESUMIDA',

    FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA STRING
        COMMENT 'Indica se as cinco pontuações foram calculadas no cenário RENDA_PRESUMIDA: S=completa; N=incompleta',

    NR_PONT_MAX_RENDA_PRESUMIDA INT
        COMMENT 'Maior pontuação final obtida no cenário RENDA_PRESUMIDA',

    QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA INT
        COMMENT 'Quantidade de temas com a maior pontuação no cenário RENDA_PRESUMIDA',

    CD_TEMA_VENCEDOR_RENDA_PRESUMIDA INT
        COMMENT 'Código do tema vencedor no cenário RENDA_PRESUMIDA: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição correspondente ao tema vencedor no cenário RENDA_PRESUMIDA',


    -- ============================================================
    -- CENÁRIO ENTRADAS REALIZADAS
    -- ============================================================

    VL_RES_ORC_ENTRADAS_REALIZADAS DECIMAL(25,2)
        COMMENT 'Resultado orçamentário recalculado no cenário ENTRADAS_REALIZADAS',

    PC_SAI_ENT_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Relação entre saídas e base utilizada no cenário ENTRADAS_REALIZADAS',

    CD_RES_ORC_ENTRADAS_REALIZADAS INT
        COMMENT 'Código do resultado orçamentário no cenário ENTRADAS_REALIZADAS: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    CD_FAIXA_ORC_ENTRADAS_REALIZADAS INT
        COMMENT 'Código da faixa orçamentária no cenário ENTRADAS_REALIZADAS: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES_ENTRADAS_REALIZADAS STRING
        COMMENT 'Intensidade do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    TX_STS_FINAL_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição final do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    PC_SAI_IND_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas no cenário ENTRADAS_REALIZADAS',

    PC_SAI_ESS_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais no cenário ENTRADAS_REALIZADAS',

    PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais no cenário ENTRADAS_REALIZADAS',

    PC_SAI_FUT_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro no cenário ENTRADAS_REALIZADAS',

    PC_SAI_OBR_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_IND_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Futuro no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Futuro no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_IND_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito, no cenário ENTRADAS_REALIZADAS',

    FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS STRING
        COMMENT 'Indica se as cinco pontuações foram calculadas no cenário ENTRADAS_REALIZADAS: S=completa; N=incompleta',

    NR_PONT_MAX_ENTRADAS_REALIZADAS INT
        COMMENT 'Maior pontuação final obtida no cenário ENTRADAS_REALIZADAS',

    QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS INT
        COMMENT 'Quantidade de temas com a maior pontuação no cenário ENTRADAS_REALIZADAS',

    CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS INT
        COMMENT 'Código do tema vencedor no cenário ENTRADAS_REALIZADAS: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição correspondente ao tema vencedor no cenário ENTRADAS_REALIZADAS'

)
COMMENT 'Resultado negocial do Radar Financeiro por cliente, contendo resultado oficial e os dois cenários calculados atualmente'
STORED AS PARQUET;
"""


In [ ]:
%%spark
inicio = time.perf_counter()
spark.sql('DROP TABLE IF EXISTS ANA_RADAR_FIN_CLI')
spark.sql(DDL_ANA_RADAR_FIN_CLI)
df_final.write.mode('append').insertInto('ANA_RADAR_FIN_CLI')
print(f'Escrita: {qt_final} clientes | {time.perf_counter() - inicio:.2f}s')
assert spark.table('ANA_RADAR_FIN_CLI').count() == qt_final, 'Quantidade publicada divergente.'
df_final.unpersist()
df_base.unpersist()
df_contexto.unpersist()
df_publico.unpersist()
df_pares_exatos.unpersist()
df_pares_borda.unpersist()
